In [1]:
import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler, normalize
from scipy.sparse import csr_matrix
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import gc

In [2]:
print("="*70)
print("PHASE 3: HYBRID RECOMMENDATION - 500K REVIEW SUBSET")
print("="*70)

# ============================================================
# STEP 1: LOAD AND SAMPLE TO 500K
# ============================================================
print("\n[1/7] Loading data and sampling to 500K reviews...")

df_train_full = pd.read_csv('train_data.csv')
print(f"  Full dataset: {len(df_train_full):,} reviews")

# Sample 500K reviews
sample_size = min(500_000, len(df_train_full))
df_train = df_train_full.sample(n=sample_size, random_state=42)
print(f"✓ Sampled: {len(df_train):,} reviews")
print(f"  Items: {df_train['item_id'].nunique():,}")
print(f"  Users: {df_train['user_id'].nunique():,}")

# Load and filter item features
item_features_full = pd.read_csv('item_features.csv')
print(f"✓ Item features: {len(item_features_full):,} items")

items_in_sample = set(df_train['item_id'].unique())
item_features = item_features_full[item_features_full['item_id'].isin(items_in_sample)].reset_index(drop=True)
print(f"✓ Filtered to: {len(item_features):,} items in sample")

# Load Word2Vec embeddings
doc_embeddings_full = np.load('doc_embeddings.npy')
with open('w2v_model.pkl', 'rb') as f:
    w2v_model = pickle.load(f)
print(f"✓ Loaded Word2Vec: {doc_embeddings_full.shape}")

# Load BERT embeddings (NEW!)
bert_embeddings_full = np.load('bert_embeddings.npy')
print(f"✓ Loaded BERT: {bert_embeddings_full.shape}")

PHASE 3: HYBRID RECOMMENDATION - 500K REVIEW SUBSET

[1/7] Loading data and sampling to 500K reviews...
  Full dataset: 1,918,055 reviews
✓ Sampled: 500,000 reviews
  Items: 110,226
  Users: 313,340
✓ Item features: 199,368 items
✓ Filtered to: 110,226 items in sample
✓ Loaded Word2Vec: (1918055, 100)
✓ Loaded BERT: (1918055, 384)


In [3]:
# ============================================================
# STEP 2: COLLABORATIVE FILTERING (SVD)
# ============================================================
print("\n[2/7] Training Collaborative Filtering (SVD)...")

user_to_idx = {uid: i for i, uid in enumerate(df_train['user_id'].unique())}
item_to_idx = {iid: i for i, iid in enumerate(df_train['item_id'].unique())}

print(f"  User-Item Matrix: ({len(user_to_idx):,}, {len(item_to_idx):,})")

row_indices = df_train['user_id'].map(user_to_idx).values
col_indices = df_train['item_id'].map(item_to_idx).values
ratings = df_train['rating'].values

user_item_matrix = csr_matrix(
    (ratings, (row_indices, col_indices)),
    shape=(len(user_to_idx), len(item_to_idx))
)

print(f"  Sparsity: {(1 - user_item_matrix.nnz / (user_item_matrix.shape[0] * user_item_matrix.shape[1])) * 100:.2f}%")

global_mean = ratings.mean()
print(f"  Global mean rating: {global_mean:.2f}")

print("  Computing biases...")
user_bias = (df_train.groupby('user_id')['rating'].mean() - global_mean).to_dict()
item_bias = (df_train.groupby('item_id')['rating'].mean() - global_mean).to_dict()

for uid in user_to_idx.keys():
    if uid not in user_bias:
        user_bias[uid] = 0.0
for iid in item_to_idx.keys():
    if iid not in item_bias:
        item_bias[iid] = 0.0

print("  Training SVD (n_components=50)...")
n_factors = 50
svd_model = TruncatedSVD(n_components=n_factors, random_state=42, n_iter=10)
user_factors = svd_model.fit_transform(user_item_matrix)
item_factors = svd_model.components_.T

print(f"✓ SVD trained")
print(f"  Explained variance: {svd_model.explained_variance_ratio_.sum():.4f}")
print(f"  User factors: {user_factors.shape}")
print(f"  Item factors: {item_factors.shape}")




[2/7] Training Collaborative Filtering (SVD)...
  User-Item Matrix: (313,340, 110,226)
  Sparsity: 100.00%
  Global mean rating: 4.23
  Computing biases...
  Training SVD (n_components=50)...
✓ SVD trained
  Explained variance: 0.0623
  User factors: (313340, 50)
  Item factors: (110226, 50)


In [4]:
# ============================================================
# STEP 3: CONTENT-BASED FEATURES (3 VERSIONS)
# ============================================================
print("\n[3/7] Building content-based features (3 versions)...")

scaler = StandardScaler()

# Sentiment
sentiment_features = item_features[['sentiment_polarity', 'sentiment_subjectivity']].fillna(0).values
sentiment_norm = scaler.fit_transform(sentiment_features)

# Topics
topic_features = item_features[[f'topic_{i}' for i in range(15)]].fillna(0).values

# Aspects
aspect_cols = ['plot', 'characters', 'writing_style', 'setting', 'emotion', 'pacing']
aspect_features = item_features[[f'aspect_{asp}' for asp in aspect_cols]].fillna(0).values

# Word2Vec embeddings (100 dims)
w2v_cols = [f'embedding_{i}' for i in range(100)]
w2v_features = item_features[w2v_cols].fillna(0).astype(np.float32).values

# BERT embeddings (384 dims) - NEW!
bert_cols = [f'bert_embedding_{i}' for i in range(384)]
bert_features = item_features[bert_cols].fillna(0).astype(np.float32).values

print(f"✓ Base features ready:")
print(f"  - Sentiment: {sentiment_norm.shape}")
print(f"  - Topics: {topic_features.shape}")
print(f"  - Aspects: {aspect_features.shape}")
print(f"  - Word2Vec: {w2v_features.shape}")
print(f"  - BERT: {bert_features.shape}")

# ============================================================
# VERSION 1: Word2Vec Only (Baseline)
# ============================================================
print("\n[3a] Version 1: Word2Vec embeddings...")
combined_w2v = np.hstack([
    sentiment_norm * 0.1,
    topic_features * 0.2,
    aspect_features * 0.3,
    w2v_features * 0.4
]).astype(np.float32)

content_scaler_w2v = StandardScaler()
content_w2v = content_scaler_w2v.fit_transform(combined_w2v)
content_w2v = normalize(content_w2v, norm='l2').astype(np.float32)
print(f"✓ Word2Vec features: {content_w2v.shape}, ~{content_w2v.nbytes / (1024**2):.1f} MB")

# ============================================================
# VERSION 2: BERT Only
# ============================================================
print("\n[3b] Version 2: BERT embeddings...")
combined_bert = np.hstack([
    sentiment_norm * 0.1,
    topic_features * 0.2,
    aspect_features * 0.3,
    bert_features * 0.4
]).astype(np.float32)

content_scaler_bert = StandardScaler()
content_bert = content_scaler_bert.fit_transform(combined_bert)
content_bert = normalize(content_bert, norm='l2').astype(np.float32)
print(f"✓ BERT features: {content_bert.shape}, ~{content_bert.nbytes / (1024**2):.1f} MB")

# ============================================================
# VERSION 3: Word2Vec + BERT Combined (30% + 70%)
# ============================================================
print("\n[3c] Version 3: Word2Vec (30%) + BERT (70%)...")
combined_both = np.hstack([
    sentiment_norm * 0.1,
    topic_features * 0.2,
    aspect_features * 0.3,
    w2v_features * 0.3 * 0.4,  # 30% of embedding weight
    bert_features * 0.7 * 0.4   # 70% of embedding weight
]).astype(np.float32)

content_scaler_both = StandardScaler()
content_both = content_scaler_both.fit_transform(combined_both)
content_both = normalize(content_both, norm='l2').astype(np.float32)
print(f"✓ Combined features: {content_both.shape}, ~{content_both.nbytes / (1024**2):.1f} MB")

# Cleanup
del sentiment_features, topic_features, aspect_features, w2v_features, bert_features
del combined_w2v, combined_bert, combined_both
gc.collect()

print(f"\n✓ All 3 feature versions ready for evaluation!")


[3/7] Building content-based features (3 versions)...
✓ Base features ready:
  - Sentiment: (110226, 2)
  - Topics: (110226, 15)
  - Aspects: (110226, 6)
  - Word2Vec: (110226, 100)
  - BERT: (110226, 384)

[3a] Version 1: Word2Vec embeddings...
✓ Word2Vec features: (110226, 123), ~51.7 MB

[3b] Version 2: BERT embeddings...
✓ BERT features: (110226, 407), ~171.1 MB

[3c] Version 3: Word2Vec (30%) + BERT (70%)...
✓ Combined features: (110226, 507), ~213.2 MB

✓ All 3 feature versions ready for evaluation!


In [5]:
# ============================================================
# STEP 4: COMPUTE ITEM SIMILARITY (3 VERSIONS)
# ============================================================
print("\n[4/7] Computing item similarity for all 3 versions...")

def compute_similarity_dict(content_features, version_name):
    """Compute similarity dictionary for a given feature set"""
    print(f"\n  [{version_name}] Processing {len(item_features):,} items...")
    
    batch_size = 1000
    n_items = len(item_features)
    similarity_dict = {}
    
    USE_TOP_K = 1000
    SIMILARITY_THRESHOLD = 0.2
    
    for i in range(0, n_items, batch_size):
        batch_end = min(i + batch_size, n_items)
        batch_features = content_features[i:batch_end]
        
        batch_similarity = cosine_similarity(batch_features, content_features)
        
        for local_idx in range(batch_similarity.shape[0]):
            global_idx = i + local_idx
            sims = batch_similarity[local_idx]
            
            valid_mask = sims > SIMILARITY_THRESHOLD
            valid_indices = np.where(valid_mask)[0]
            valid_sims = sims[valid_indices]
            
            if len(valid_sims) > 0:
                if len(valid_sims) > USE_TOP_K:
                    top_k_indices = np.argsort(valid_sims)[-USE_TOP_K:]
                    valid_indices = valid_indices[top_k_indices]
                    valid_sims = valid_sims[top_k_indices]
                
                sort_order = np.argsort(valid_sims)[::-1]
                
                similarity_dict[global_idx] = {
                    'indices': valid_indices[sort_order].astype(np.int32),
                    'similarities': valid_sims[sort_order].astype(np.float32)
                }
        
        if (i + batch_size) % 10000 == 0 or batch_end == n_items:
            avg_sims = np.mean([len(v['indices']) for v in similarity_dict.values()]) if similarity_dict else 0
            print(f"    ✓ {batch_end:,}/{n_items:,} (avg: {avg_sims:.0f} sims)")
        
        del batch_similarity
        gc.collect()
    
    total_mem = (sum(len(v['indices']) for v in similarity_dict.values()) * 8) / (1024**2)
    print(f"  ✓ {version_name}: {total_mem:.1f} MB")
    return similarity_dict

# Compute all 3 versions
sim_dict_w2v = compute_similarity_dict(content_w2v, "Word2Vec")
sim_dict_bert = compute_similarity_dict(content_bert, "BERT")
sim_dict_both = compute_similarity_dict(content_both, "Combined")

print(f"\n✓ All similarity indices created!")

# Cleanup content features (keep similarity dicts)
del content_w2v, content_bert, content_both
gc.collect()


[4/7] Computing item similarity for all 3 versions...

  [Word2Vec] Processing 110,226 items...
    ✓ 10,000/110,226 (avg: 1000 sims)
    ✓ 20,000/110,226 (avg: 1000 sims)
    ✓ 30,000/110,226 (avg: 1000 sims)
    ✓ 40,000/110,226 (avg: 1000 sims)
    ✓ 50,000/110,226 (avg: 1000 sims)
    ✓ 60,000/110,226 (avg: 1000 sims)
    ✓ 70,000/110,226 (avg: 1000 sims)
    ✓ 80,000/110,226 (avg: 1000 sims)
    ✓ 90,000/110,226 (avg: 1000 sims)
    ✓ 100,000/110,226 (avg: 1000 sims)
    ✓ 110,000/110,226 (avg: 1000 sims)
    ✓ 110,226/110,226 (avg: 1000 sims)
  ✓ Word2Vec: 841.0 MB

  [BERT] Processing 110,226 items...
    ✓ 10,000/110,226 (avg: 999 sims)
    ✓ 20,000/110,226 (avg: 999 sims)
    ✓ 30,000/110,226 (avg: 999 sims)
    ✓ 40,000/110,226 (avg: 999 sims)
    ✓ 50,000/110,226 (avg: 999 sims)
    ✓ 60,000/110,226 (avg: 999 sims)
    ✓ 70,000/110,226 (avg: 999 sims)
    ✓ 80,000/110,226 (avg: 999 sims)
    ✓ 90,000/110,226 (avg: 999 sims)
    ✓ 100,000/110,226 (avg: 999 sims)
    ✓ 110,00

0

In [7]:
# ============================================================
# CHECKPOINT: SAVE INTERMEDIATE RESULTS (STEPS 2-4)
# ============================================================
print("\n[4.5/7] Saving intermediate results (checkpointing)...")

joblib.dump(user_factors, 'user_factors_500k.pkl', compress=3)
print("  ✓ user_factors_500k.pkl")

joblib.dump(item_factors, 'item_factors_500k.pkl', compress=3)
print("  ✓ item_factors_500k.pkl")

joblib.dump(user_to_idx, 'user_to_idx_500k.pkl', compress=3)
print("  ✓ user_to_idx_500k.pkl")

joblib.dump(item_to_idx, 'item_to_idx_500k.pkl', compress=3)
print("  ✓ item_to_idx_500k.pkl")

joblib.dump(user_bias, 'user_bias_500k.pkl', compress=3)
print("  ✓ user_bias_500k.pkl")

joblib.dump(item_bias, 'item_bias_500k.pkl', compress=3)
print("  ✓ item_bias_500k.pkl")

#joblib.dump(item_similarity_dict, 'item_similarity_dict_500k.pkl', compress=3)
#print("  ✓ item_similarity_dict_500k.pkl")

joblib.dump({'global_mean': global_mean}, 'global_stats_500k.pkl', compress=3)
print("  ✓ global_stats_500k.pkl")

print("✓ Checkpoint saved! Can resume from here if needed.")



[4.5/7] Saving intermediate results (checkpointing)...
  ✓ user_factors_500k.pkl
  ✓ item_factors_500k.pkl
  ✓ user_to_idx_500k.pkl
  ✓ item_to_idx_500k.pkl
  ✓ user_bias_500k.pkl
  ✓ item_bias_500k.pkl
  ✓ global_stats_500k.pkl
✓ Checkpoint saved! Can resume from here if needed.


In [10]:
# ============================================================
# STEP 5: HYBRID RECOMMENDER CLASS
# ============================================================
print("\n[5/7] Building hybrid recommender...")

class HybridRecommender:
    def __init__(self, user_factors, item_factors, user_to_idx, item_to_idx,
                 item_similarity_dict, item_features, df_train,
                 user_bias, item_bias, global_mean, alpha=0.5):
        
        self.user_factors = user_factors
        self.item_factors = item_factors
        self.user_to_idx = user_to_idx
        self.item_to_idx = item_to_idx
        self.item_similarity_dict = item_similarity_dict
        self.item_features = item_features
        self.user_bias = user_bias
        self.item_bias = item_bias
        self.global_mean = global_mean
        self.alpha = alpha
        
        self.idx_to_item = {v: k for k, v in item_to_idx.items()}
        self.user_items = df_train.groupby('user_id')['item_id'].apply(set).to_dict()
        
        self.user_item_ratings = {}
        for uid in self.user_to_idx.keys():
            self.user_item_ratings[uid] = {}
        
        for _, row in df_train.iterrows():
            uid = row['user_id']
            iid = row['item_id']
            if iid in item_to_idx:
                self.user_item_ratings[uid][self.item_to_idx[iid]] = row['rating']
    
    def get_cf_score(self, user_id, item_id):
        """CF score with bias adjustment"""
        if user_id not in self.user_to_idx or item_id not in self.item_to_idx:
            return self.global_mean
        
        u_idx = self.user_to_idx[user_id]
        i_idx = self.item_to_idx[item_id]
        
        dot_product = np.dot(self.user_factors[u_idx], self.item_factors[i_idx])
        prediction = (self.global_mean + 
                     self.user_bias.get(user_id, 0.0) + 
                     self.item_bias.get(item_id, 0.0) + 
                     dot_product)
        
        return np.clip(prediction, 1.0, 5.0)
    
    def get_content_score(self, user_id, item_id):
        """Content-based score using similarities"""
        if user_id not in self.user_to_idx or item_id not in self.item_to_idx:
            return self.global_mean
        
        item_idx = self.item_to_idx[item_id]
        user_rated_items = self.user_items.get(user_id, set())
        
        if not user_rated_items or item_idx not in self.item_similarity_dict:
            return self.global_mean
        
        user_item_indices = {self.item_to_idx[iid]: iid 
                            for iid in user_rated_items 
                            if iid in self.item_to_idx}
        
        if not user_item_indices:
            return self.global_mean
        
        similar_data = self.item_similarity_dict[item_idx]
        similar_indices = similar_data['indices']
        similar_sims = similar_data['similarities'].astype(np.float32)
        
        weighted_sum = 0.0
        sim_sum = 0.0
        
        for sim_idx, sim_val in zip(similar_indices, similar_sims):
            if sim_idx in user_item_indices:
                rating = self.user_item_ratings[user_id].get(sim_idx, self.global_mean)
                weighted_sum += float(sim_val) * rating
                sim_sum += float(sim_val)
        
        if sim_sum == 0:
            return self.global_mean
        
        prediction = weighted_sum / sim_sum
        return np.clip(prediction, 1.0, 5.0)

# DON'T initialize recommender yet - we'll do it in Step 6 after evaluating all versions
print("✓ HybridRecommender class defined")
print("  (Will initialize after evaluation)")


[5/7] Building hybrid recommender...
✓ HybridRecommender class defined
  (Will initialize after evaluation)


In [12]:
# ============================================================
# STEP 7: SAVE MODELS (ALL 3 VERSIONS)
# ============================================================
print("\n[7/7] Saving models...")

# Save CF components
joblib.dump(user_factors, 'user_factors_500k.pkl', compress=3)
joblib.dump(item_factors, 'item_factors_500k.pkl', compress=3)
joblib.dump(user_to_idx, 'user_to_idx_500k.pkl', compress=3)
joblib.dump(item_to_idx, 'item_to_idx_500k.pkl', compress=3)
joblib.dump(user_bias, 'user_bias_500k.pkl', compress=3)
joblib.dump(item_bias, 'item_bias_500k.pkl', compress=3)

# Save all 3 similarity dictionaries
joblib.dump(sim_dict_w2v, 'item_similarity_w2v_500k.pkl', compress=3)
print("✓ Word2Vec similarity dict")

joblib.dump(sim_dict_bert, 'item_similarity_bert_500k.pkl', compress=3)
print("✓ BERT similarity dict")

joblib.dump(sim_dict_both, 'item_similarity_both_500k.pkl', compress=3)
print("✓ Combined similarity dict")

# Save recommender with best similarity dict
best_sim_dict = {
    'Word2Vec': sim_dict_w2v,
    'BERT': sim_dict_bert,
    'Combined': sim_dict_both
}[embedding_comparison_results['best_method']]

recommender_best = HybridRecommender(
    user_factors, item_factors, user_to_idx, item_to_idx,
    best_sim_dict, item_features, df_train,
    user_bias, item_bias, global_mean, alpha=0.6
)
joblib.dump(recommender_best, 'hybrid_recommender_500k.pkl', compress=3)
print(f"✓ Recommender (using {embedding_comparison_results['best_method']} embeddings)")

# Save comparison results
joblib.dump(embedding_comparison_results, 'embedding_comparison_500k.pkl', compress=3)
print("✓ Embedding comparison results")

print(f"\n{'='*70}")
print("PHASE 3 COMPLETE!")
print(f"{'='*70}")
print(f"\n✅ Evaluated 3 embedding approaches:")
print(f"   • Word2Vec (100-dim, domain-specific)")
print(f"   • BERT (384-dim, pre-trained)")
print(f"   • Combined (30% Word2Vec + 70% BERT)")
print(f"\n🏆 Best method: {embedding_comparison_results['best_method']}")
print(f"   RMSE: {embedding_comparison_results['best_rmse']:.4f}")
print(f"   Improvement: {((cf_rmse - embedding_comparison_results['best_rmse'])/cf_rmse*100):.1f}%")



[7/7] Saving models...
✓ Word2Vec similarity dict
✓ BERT similarity dict
✓ Combined similarity dict
✓ Recommender (using Combined embeddings)
✓ Embedding comparison results

PHASE 3 COMPLETE!

✅ Evaluated 3 embedding approaches:
   • Word2Vec (100-dim, domain-specific)
   • BERT (384-dim, pre-trained)
   • Combined (30% Word2Vec + 70% BERT)

🏆 Best method: Combined
   RMSE: 0.3530
   Improvement: 38.7%


In [14]:
"""
Improved Hybrid Recommender Evaluation:
- Tests all 3 embedding versions on validation set
- Variance scaling
- Adaptive alpha
"""

import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import joblib
import warnings
warnings.filterwarnings('ignore')

print("="*70)
print("IMPROVED HYBRID RECOMMENDER - VALIDATION (ALL 3 EMBEDDINGS)")
print("="*70)

# ============================================================
# LOAD MODEL AND DATA
# ============================================================
print("\n[1/3] Loading models and validation data...")

# Load all 3 similarity dictionaries
sim_dict_w2v = joblib.load('item_similarity_w2v_500k.pkl')
sim_dict_bert = joblib.load('item_similarity_bert_500k.pkl')
sim_dict_both = joblib.load('item_similarity_both_500k.pkl')

# Load CF components
user_factors = joblib.load('user_factors_500k.pkl')
item_factors = joblib.load('item_factors_500k.pkl')
user_to_idx = joblib.load('user_to_idx_500k.pkl')
item_to_idx = joblib.load('item_to_idx_500k.pkl')
user_bias = joblib.load('user_bias_500k.pkl')
item_bias = joblib.load('item_bias_500k.pkl')
global_stats = joblib.load('global_stats_500k.pkl')
global_mean = global_stats['global_mean']

# Load item features
item_features_full = pd.read_csv('item_features.csv')
items_in_idx = set(item_to_idx.keys())
item_features = item_features_full[item_features_full['item_id'].isin(items_in_idx)].reset_index(drop=True)

# Load training data for user_items lookup
df_train = pd.read_csv('train_data.csv')
df_train = df_train.sample(n=500_000, random_state=42)

# Load validation data
df_val = pd.read_csv('val_data.csv')
df_val_filtered = df_val[
    (df_val['user_id'].isin(user_to_idx.keys())) & 
    (df_val['item_id'].isin(item_to_idx.keys()))
].copy()

sample_size = min(10000, len(df_val_filtered))
test_sample = df_val_filtered.sample(n=sample_size, random_state=42)

print(f"✓ Loaded {sample_size:,} validation samples")

# Build lookups
user_items_lookup = df_train.groupby('user_id')['item_id'].apply(set).to_dict()
user_item_ratings = {}
for uid in user_to_idx.keys():
    user_item_ratings[uid] = {}
for _, row in df_train.iterrows():
    uid, iid = row['user_id'], row['item_id']
    if iid in item_to_idx:
        user_item_ratings[uid][item_to_idx[iid]] = row['rating']

user_activity = {uid: len(user_item_ratings.get(uid, {})) for uid in user_to_idx.keys()}

# ============================================================
# EVALUATION FUNCTIONS
# ============================================================

def get_cf_score(user_id, item_id):
    if user_id not in user_to_idx or item_id not in item_to_idx:
        return global_mean
    u_idx, i_idx = user_to_idx[user_id], item_to_idx[item_id]
    dot = np.dot(user_factors[u_idx], item_factors[i_idx])
    pred = global_mean + user_bias.get(user_id, 0.0) + item_bias.get(item_id, 0.0) + dot
    return np.clip(pred, 1.0, 5.0)

def get_content_score(user_id, item_id, sim_dict):
    if user_id not in user_to_idx or item_id not in item_to_idx:
        return global_mean
    
    item_idx = item_to_idx[item_id]
    user_rated = user_items_lookup.get(user_id, set())
    
    if not user_rated or item_idx not in sim_dict:
        return global_mean
    
    user_item_indices = {item_to_idx[iid]: iid for iid in user_rated if iid in item_to_idx}
    if not user_item_indices:
        return global_mean
    
    similar_data = sim_dict[item_idx]
    weighted_sum, sim_sum = 0.0, 0.0
    
    for sim_idx, sim_val in zip(similar_data['indices'], similar_data['similarities']):
        if sim_idx in user_item_indices:
            rating = user_item_ratings[user_id].get(sim_idx, global_mean)
            weighted_sum += float(sim_val) * rating
            sim_sum += float(sim_val)
    
    return np.clip(weighted_sum / sim_sum, 1.0, 5.0) if sim_sum > 0 else global_mean

# ============================================================
# EVALUATE ALL 3 EMBEDDINGS
# ============================================================
print("\n[2/3] Evaluating all 3 embeddings on validation set...")

actual = test_sample['rating'].tolist()
cf_preds = [get_cf_score(row['user_id'], row['item_id']) for _, row in test_sample.iterrows()]

embeddings = {
    'Word2Vec': sim_dict_w2v,
    'BERT': sim_dict_bert,
    'Combined': sim_dict_both
}

results_validation = {'CF Only': np.sqrt(mean_squared_error(actual, cf_preds))}

for name, sim_dict in embeddings.items():
    print(f"  [{name}]")
    content_preds = [get_content_score(row['user_id'], row['item_id'], sim_dict) 
                     for _, row in test_sample.iterrows()]
    hybrid_preds = [0.6*cf + 0.4*cont for cf, cont in zip(cf_preds, content_preds)]
    rmse = np.sqrt(mean_squared_error(actual, hybrid_preds))
    results_validation[f'Hybrid ({name})'] = rmse

print("\n" + "="*70)
print("VALIDATION RESULTS (ALL 3 EMBEDDINGS)")
print("="*70)
print("\nMethod              | RMSE   | vs CF")
print("--------------------|--------|-------")
baseline = results_validation['CF Only']
for method, rmse in results_validation.items():
    improvement = ((baseline - rmse) / baseline) * 100
    print(f"{method:19} | {rmse:.4f} | {improvement:+.1f}%")

best = min([(k, v) for k, v in results_validation.items() if k != 'CF Only'], key=lambda x: x[1])
print(f"\n🏆 Best on validation: {best[0]} (RMSE = {best[1]:.4f})")

# ============================================================
# USER SEGMENTATION
# ============================================================
print("\n[3/3] Analyzing by user activity...")

test_sample['user_activity'] = test_sample['user_id'].map(user_activity)
segments = [
    ('Sparse (1-10)', 1, 10),
    ('Medium (11-30)', 11, 30),
    ('Active (31+)', 31, 1000)
]

print("\n" + "="*70)
print("PERFORMANCE BY USER ACTIVITY")
print("="*70)
print("\nSegment         | Size  | CF    | Word2Vec | BERT  | Combined")
print("----------------|-------|-------|----------|-------|----------")

# Pre-compute all predictions
all_preds = {
    'CF': cf_preds,
    'Word2Vec': [],
    'BERT': [],
    'Combined': []
}

# Compute hybrid predictions for each embedding
for i, (_, row) in enumerate(test_sample.iterrows()):
    cf = cf_preds[i]
    
    cont_w2v = get_content_score(row['user_id'], row['item_id'], sim_dict_w2v)
    all_preds['Word2Vec'].append(0.6 * cf + 0.4 * cont_w2v)
    
    cont_bert = get_content_score(row['user_id'], row['item_id'], sim_dict_bert)
    all_preds['BERT'].append(0.6 * cf + 0.4 * cont_bert)
    
    cont_both = get_content_score(row['user_id'], row['item_id'], sim_dict_both)
    all_preds['Combined'].append(0.6 * cf + 0.4 * cont_both)

for seg_name, min_act, max_act in segments:
    seg_mask = (test_sample['user_activity'] >= min_act) & (test_sample['user_activity'] <= max_act)
    seg_indices = test_sample[seg_mask].index.tolist()
    
    if len(seg_indices) == 0:
        continue
    
    seg_actual = [actual[i] for i, idx in enumerate(test_sample.index) if idx in seg_indices]
    seg_rmses = {}
    
    for method in ['CF', 'Word2Vec', 'BERT', 'Combined']:
        seg_preds = [all_preds[method][i] for i, idx in enumerate(test_sample.index) if idx in seg_indices]
        seg_rmses[method] = np.sqrt(mean_squared_error(seg_actual, seg_preds))
    
    print(f"{seg_name:15} | {len(seg_indices):5} | {seg_rmses['CF']:.3f} | {seg_rmses['Word2Vec']:.4f}   | {seg_rmses['BERT']:.3f} | {seg_rmses['Combined']:.4f}")

# Save results
validation_results = {
    'overall': results_validation,
    'best_method': best[0],
    'best_rmse': best[1],
    'segment_analysis': segments
}

joblib.dump(validation_results, 'validation_results_TRUE.pkl')
print(f"\n✓ Validation results saved to 'validation_results_TRUE.pkl'")

print("\n" + "="*70)
print("VALIDATION COMPLETE!")
print("="*70)
print("\n✅ All 3 embeddings evaluated on validation set")
print(f"🏆 Best: {best[0]} with RMSE = {best[1]:.4f}")
print("✅ Ready for Phase 4 (test set evaluation)!")

IMPROVED HYBRID RECOMMENDER - VALIDATION (ALL 3 EMBEDDINGS)

[1/3] Loading models and validation data...
✓ Loaded 10,000 validation samples

[2/3] Evaluating all 3 embeddings on validation set...
  [Word2Vec]
  [BERT]
  [Combined]

VALIDATION RESULTS (ALL 3 EMBEDDINGS)

Method              | RMSE   | vs CF
--------------------|--------|-------
CF Only             | 0.9728 | +0.0%
Hybrid (Word2Vec)   | 0.9065 | +6.8%
Hybrid (BERT)       | 0.9066 | +6.8%
Hybrid (Combined)   | 0.9062 | +6.8%

🏆 Best on validation: Hybrid (Combined) (RMSE = 0.9062)

[3/3] Analyzing by user activity...

PERFORMANCE BY USER ACTIVITY

Segment         | Size  | CF    | Word2Vec | BERT  | Combined
----------------|-------|-------|----------|-------|----------
Sparse (1-10)   |  7390 | 0.990 | 0.9174   | 0.916 | 0.9161
Medium (11-30)  |  1392 | 0.942 | 0.9023   | 0.908 | 0.9075
Active (31+)    |  1185 | 0.906 | 0.8516   | 0.853 | 0.8514

✓ Validation results saved to 'validation_results_TRUE.pkl'

VALIDATION COM